# 02 Sanity Check WRDS Extracts

This notebook is a lightweight research-facing driver for `sanity_check_wrds_extract.py`.

The production logic lives in the `.py` script so the same checks can be called from `01_run_wrds_extracts.ipynb`, the terminal, or future pipeline automation. Use this notebook when you want to rerun QA interactively and inspect the generated tables.

## Run sanity checks

This validates the raw membership, CRSP panel, FF3, and SPY benchmark extract files, then saves CSV/TXT/PNG outputs under `data/sanity_checks`.

In [1]:
%run ./sanity_check_wrds_extract.py --project-root . --raw-dir data/raw --benchmark-dir data/benchmark --outdir data/sanity_checks

2026-05-01 17:56:57 INFO Loading /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/raw/sp500_membership_monthly.parquet
2026-05-01 17:56:57 INFO Loading /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/raw/crsp_monthly_sp500_panel.parquet
2026-05-01 17:56:57 INFO Loading /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/raw/ff3_monthly.parquet
2026-05-01 17:56:57 INFO Loading /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/benchmark/spy_monthly.parquet
2026-05-01 17:56:57 INFO Saved sanity-check tables and report to: /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/sanity_checks
2026-05-01 17:56:57 INFO Saved plot: /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/sanity_checks/membership_monthly_count.png
2026-05-01 17:56:57 INFO Saved plot: /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/da

WRDS extract sanity check

Membership
- month_end range: 2000-01-31 to 2024-12-31
- rows: 150,492
- unique permnos: 1,062
- monthly constituent count stats: count=300, mean=501.64, std=2.27, min=496.00, p25=500.00, p50=500.00, p75=504.00, max=506.00

CRSP panel
- month_end range: 2000-01-31 to 2024-12-31
- rows: 150,900
- unique permnos: 1,062
- duplicate (permno, month_end) keys: YES
- duplicate rows involved: 804
- duplicate key count: 396
- monthly CRSP count stats: count=300, mean=501.64, std=2.27, min=496.00, p25=500.00, p50=500.00, p75=504.00, max=506.00
- CRSP minus membership count stats: count=300, mean=0.00, std=0.00, min=0.00, p25=0.00, p50=0.00, p75=0.00, max=0.00

Fama-French factors
- month_end range: 2000-01-31 to 2025-12-31
- rows: 312
- unique month_end: YES
- duplicate month rows involved: 0
- duplicate month count: 0

SPY benchmark
- month_end range: 2000-01-31 to 2024-12-31
- rows: 300
- unique month_end: YES
- duplicate month rows involved: 0
- duplicate month coun

## Inspect saved outputs

These cells read the artifacts generated by the script. They are intentionally separate from the check logic so the script stays the single source of truth.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

outdir = Path("data/sanity_checks")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [3]:
print((outdir / "coverage_summary.txt").read_text())

WRDS extract sanity check

Membership
- month_end range: 2000-01-31 to 2024-12-31
- rows: 150,492
- unique permnos: 1,062
- monthly constituent count stats: count=300, mean=501.64, std=2.27, min=496.00, p25=500.00, p50=500.00, p75=504.00, max=506.00

CRSP panel
- month_end range: 2000-01-31 to 2024-12-31
- rows: 150,900
- unique permnos: 1,062
- duplicate (permno, month_end) keys: YES
- duplicate rows involved: 804
- duplicate key count: 396
- monthly CRSP count stats: count=300, mean=501.64, std=2.27, min=496.00, p25=500.00, p50=500.00, p75=504.00, max=506.00
- CRSP minus membership count stats: count=300, mean=0.00, std=0.00, min=0.00, p25=0.00, p50=0.00, p75=0.00, max=0.00

Fama-French factors
- month_end range: 2000-01-31 to 2025-12-31
- rows: 312
- unique month_end: YES
- duplicate month rows involved: 0
- duplicate month count: 0

SPY benchmark
- month_end range: 2000-01-31 to 2024-12-31
- rows: 300
- unique month_end: YES
- duplicate month rows involved: 0
- duplicate month coun

In [4]:
for filename in [
    "extract_dataset_summary.csv",
    "final_checks.csv",
    "missingness_summary.csv",
    "duplicate_key_summary.csv",
    "factor_coverage_summary.csv",
]:
    print(filename)
    display(pd.read_csv(outdir / filename))

extract_dataset_summary.csv


,dataset,rows,unique_permnos,month_start,month_end
0,membership,150492,"1,062.0000",2000-01-31,2024-12-31
1,crsp,150900,"1,062.0000",2000-01-31,2024-12-31
2,ff3,312,NaN,2000-01-31,2025-12-31
3,spy,300,1.0000,2000-01-31,2024-12-31


final_checks.csv


,check,value
0,"CRSP duplicate (permno, month_end) keys",396
1,FF3 duplicate month_end keys,0
2,SPY duplicate month_end keys,0
3,CRSP months missing FF3 factor data,0
4,Max absolute CRSP-minus-membership count gap,0


missingness_summary.csv


,dataset,column,missing_count,total_rows,missing_rate
0,crsp,ret,88,150900,0.0006
1,crsp,retx,88,150900,0.0006
2,crsp,dlret,150842,150900,0.9996
3,crsp,prc,13,150900,0.0001
4,crsp,shrout,0,150900,0.0000
5,crsp,vol,0,150900,0.0000
6,crsp,me,13,150900,0.0001
7,crsp,retadj,75,150900,0.0005
8,ff3,mktrf,0,312,0.0000
9,ff3,smb,0,312,0.0000


duplicate_key_summary.csv


,dataset,key,has_duplicates,duplicate_rows,duplicate_keys
0,crsp,"permno, month_end",True,804,396
1,ff3,month_end,False,0,0
2,spy,month_end,False,0,0


factor_coverage_summary.csv


,crsp_distinct_months,ff3_distinct_months,matched_crsp_months_with_ff3,missing_crsp_months_with_ff3
0,300,312,300,0


## Monthly count diagnostics

The count tables are often the fastest way to spot extraction coverage issues before building the modeling panel.

In [5]:
counts = pd.read_csv(outdir / "crsp_monthly_counts.csv", parse_dates=["month_end"])

display(counts.head())
display(counts.tail())
display(counts[["membership_count", "crsp_count", "crsp_minus_membership"]].describe())

,month_end,crsp_count,membership_count,crsp_minus_membership
0,2000-01-31,500,500,0
1,2000-02-29,500,500,0
2,2000-03-31,500,500,0
3,2000-04-30,500,500,0
4,2000-05-31,500,500,0


,month_end,crsp_count,membership_count,crsp_minus_membership
295,2024-08-31,503,503,0
296,2024-09-30,504,504,0
297,2024-10-31,503,503,0
298,2024-11-30,503,503,0
299,2024-12-31,503,503,0


,membership_count,crsp_count,crsp_minus_membership
count,300.0000,300.0000,300.0000
mean,501.6400,501.6400,0.0000
std,2.2688,2.2688,0.0000
min,496.0000,496.0000,0.0000
25%,500.0000,500.0000,0.0000
50%,500.0000,500.0000,0.0000
75%,504.0000,504.0000,0.0000
max,506.0000,506.0000,0.0000


In [6]:
largest_count_gaps = counts.assign(
    abs_gap=counts["crsp_minus_membership"].abs()
).sort_values(["abs_gap", "month_end"], ascending=[False, True])

display(largest_count_gaps.head(20))

,month_end,crsp_count,membership_count,crsp_minus_membership,abs_gap
0,2000-01-31,500,500,0,0
1,2000-02-29,500,500,0,0
2,2000-03-31,500,500,0,0
3,2000-04-30,500,500,0,0
4,2000-05-31,500,500,0,0
5,2000-06-30,500,500,0,0
6,2000-07-31,500,500,0,0
7,2000-08-31,500,500,0,0
8,2000-09-30,499,499,0,0
9,2000-10-31,500,500,0,0
